# v7 — Simple, Capable Backbone: tree model + Ridge on raw15 + bounded x9³

A fresh, minimal model that distills every lesson from v1–v6 into the simplest possible pipeline:

1. **Features:** the 15 raw columns, median-imputed, plus ONE engineered feature — the bounded cube
   `sign(x9)·min(|x9|,10)³` (the v6 discovery, r=0.45 with target).
2. **Training target:** raw `y` clipped to ±2,000 (the v3 clip-stabilization trick). Scored on raw `y`.
3. **Selection statistic:** per-fold **median** raw R² (the statistic that tracked the real leaderboard),
   with per-fold min as the private-LB risk gauge.
4. **Output calibration:** final predictions scaled to the proven leaderboard sweet spot (±800 range).

Same CV protocol as all prior versions: 5-fold, shuffled, `random_state=42`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RNG = 42
DATA_DIR = ".."

train = pd.read_csv(f"{DATA_DIR}/spring2026_kaggle_linear_regression_challenge_train.csv")
test = pd.read_csv(f"{DATA_DIR}/spring2026_kaggle_linear_regression_challenge_test.csv")

FEATS = [f"x{i}" for i in range(15)]
y_raw = train["target"].values

def build_features(df, medians):
    X = df[FEATS].fillna(medians).copy()
    x9 = X["x9"].values
    X["x9_cube_b"] = np.sign(x9) * np.minimum(np.abs(x9), 10.0) ** 3
    return X

medians = train[FEATS].median()
X = build_features(train, medians)
X_test = build_features(test, medians)

print(f"train {X.shape}, test {X_test.shape}")
print(f"target: median {np.median(y_raw):.2f}, range [{y_raw.min():.0f}, {y_raw.max():.0f}]")
print(f"corr(x9_cube_b, y) = {np.corrcoef(X['x9_cube_b'], y_raw)[0,1]:.3f}")

train (2500, 16), test (2500, 16)
target: median -0.77, range [-41008, 69628]
corr(x9_cube_b, y) = 0.192


## CV protocol

Train on the **clipped** target (±2,000), score every fold on the **raw** target. Report the
per-fold distribution — mean, **median** (leaderboard proxy), and min (private-LB risk).

In [2]:
CLIP = 2000.0
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)
FOLDS = list(kf.split(X))

def cv_per_fold(make_model, X=X, clip=CLIP):
    """5-fold CV: train on clipped target, score raw R² per fold.
    Returns (fold_scores, oof_predictions)."""
    Xv = X.values
    oof = np.zeros(len(Xv))
    scores = []
    for tr, va in FOLDS:
        model = make_model()
        model.fit(Xv[tr], np.clip(y_raw[tr], -clip, clip))
        p = model.predict(Xv[va])
        oof[va] = p
        scores.append(r2_score(y_raw[va], p))
    return np.array(scores), oof

def report(name, scores):
    print(f"{name:34s} mean {scores.mean():+.4f}  median {np.median(scores):+.4f}  "
          f"min {scores.min():+.4f}  folds {np.round(scores, 3)}")

## Model comparison

Five simple candidates, identical features and CV. Tree models capture the x9 curve and
interactions on their own; Ridge is the stable linear reference.

In [3]:
CANDIDATES = {
    "Ridge a=1000 (scaled)": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1000)),
    "RandomForest 400": lambda: RandomForestRegressor(
        n_estimators=400, min_samples_leaf=20, max_features=0.5, random_state=RNG, n_jobs=-1),
    "ExtraTrees 400": lambda: ExtraTreesRegressor(
        n_estimators=400, min_samples_leaf=20, max_features=0.5, random_state=RNG, n_jobs=-1),
    "HistGBM d3 lr.05": lambda: HistGradientBoostingRegressor(
        max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=20, random_state=RNG),
    "HistGBM d2 lr.03": lambda: HistGradientBoostingRegressor(
        max_depth=2, learning_rate=0.03, max_iter=400, min_samples_leaf=20, random_state=RNG),
}

results = {}
for name, mk in CANDIDATES.items():
    scores, oof = cv_per_fold(mk)
    results[name] = (scores, oof)
    report(name, scores)

Ridge a=1000 (scaled)              mean +0.0308  median +0.0355  min +0.0119  folds [0.018 0.012 0.038 0.05  0.035]


RandomForest 400                   mean +0.0336  median +0.0440  min +0.0152  folds [0.016 0.015 0.047 0.046 0.044]


ExtraTrees 400                     mean +0.0313  median +0.0402  min +0.0116  folds [0.016 0.012 0.04  0.047 0.042]


HistGBM d3 lr.05                   mean +0.0132  median +0.0178  min -0.0309  folds [ 0.011  0.018 -0.031  0.026  0.042]


HistGBM d2 lr.03                   mean +0.0171  median +0.0154  min -0.0121  folds [ 0.015  0.015 -0.012  0.03   0.038]


## Blend: best tree model + Ridge

A linear component hedges the private leaderboard (v3 lesson). Sweep the blend weight and
select by per-fold **median**, requiring a non-catastrophic min fold.

In [4]:
tree_names = [n for n in results if "Ridge" not in n]
best_tree = max(tree_names, key=lambda n: np.median(results[n][0]))
print(f"best tree model: {best_tree}\n")

oof_tree = results[best_tree][1]
oof_ridge = results["Ridge a=1000 (scaled)"][1]

def fold_scores(oof):
    return np.array([r2_score(y_raw[va], oof[va]) for _, va in FOLDS])

blend_rows = []
for w in [1.0, 0.8, 0.7, 0.6, 0.5, 0.4]:
    oof_b = w * oof_tree + (1 - w) * oof_ridge
    s = fold_scores(oof_b)
    blend_rows.append((w, s))
    report(f"blend w_tree={w:.1f}", s)

# select by median, tie-broken toward the safer (higher-min) option
best_w, best_s = max(blend_rows, key=lambda r: (np.median(r[1]).round(4), r[1].min()))
print(f"\nselected blend weight: {best_w} (median {np.median(best_s):+.4f}, min {best_s.min():+.4f})")
oof_blend = best_w * oof_tree + (1 - best_w) * oof_ridge

best tree model: RandomForest 400

blend w_tree=1.0                   mean +0.0336  median +0.0440  min +0.0152  folds [0.016 0.015 0.047 0.046 0.044]
blend w_tree=0.8                   mean +0.0338  median +0.0426  min +0.0146  folds [0.016 0.015 0.048 0.047 0.043]
blend w_tree=0.7                   mean +0.0337  median +0.0418  min +0.0143  folds [0.017 0.014 0.048 0.048 0.042]
blend w_tree=0.6                   mean +0.0336  median +0.0410  min +0.0140  folds [0.017 0.014 0.048 0.048 0.041]
blend w_tree=0.5                   mean +0.0333  median +0.0402  min +0.0136  folds [0.017 0.014 0.047 0.049 0.04 ]
blend w_tree=0.4                   mean +0.0330  median +0.0393  min +0.0133  folds [0.017 0.013 0.046 0.049 0.039]

selected blend weight: 1.0 (median +0.0440, min +0.0152)


## Output-scale calibration

The leaderboard punished prediction ranges beyond ~±800 (v4/Phase-9 lesson: real test extremes are
unpredictable noise). Check how per-fold median responds to scaling, then calibrate the final test
predictions to the proven ±800 sweet spot.

In [5]:
for s in [0.5, 0.7, 0.85, 1.0, 1.15, 1.3]:
    sc = fold_scores(s * oof_blend)
    report(f"scale s={s:.2f} (range ±{np.abs(s*oof_blend).max():.0f})", sc)

scale s=0.50 (range ±232)          mean +0.0249  median +0.0229  min +0.0079  folds [0.008 0.008 0.059 0.027 0.023]
scale s=0.70 (range ±325)          mean +0.0306  median +0.0320  min +0.0109  folds [0.012 0.011 0.063 0.036 0.032]
scale s=0.85 (range ±395)          mean +0.0329  median +0.0382  min +0.0131  folds [0.014 0.013 0.058 0.041 0.038]
scale s=1.00 (range ±465)          mean +0.0336  median +0.0440  min +0.0152  folds [0.016 0.015 0.047 0.046 0.044]
scale s=1.15 (range ±534)          mean +0.0327  median +0.0283  min +0.0173  folds [0.018 0.017 0.028 0.051 0.049]
scale s=1.30 (range ±604)          mean +0.0301  median +0.0194  min +0.0033  folds [0.019 0.019 0.003 0.054 0.054]


## Final fit & submissions

Refit the selected model on the full training set (clipped target) and predict the test set.

Two entries are saved, reflecting the CV-vs-leaderboard tension:

- **`submission_v7.csv` (primary):** natural scale — the per-fold scale sweep above peaks at s=1.0,
  so this is the CV-honest entry.
- **`submission_v7_sweet800.csv` (optional second):** amplified to the ±800 range where the real
  leaderboard peaked in v4–v6 testing (±800 scored 0.04623 > the ±464 v3 blend). Use it to test
  whether the LB range-preference carries over to this backbone.

In [6]:
y_clip = np.clip(y_raw, -CLIP, CLIP)

tree_full = CANDIDATES[best_tree]()
tree_full.fit(X.values, y_clip)
ridge_full = CANDIDATES["Ridge a=1000 (scaled)"]()
ridge_full.fit(X.values, y_clip)

pred = best_w * tree_full.predict(X_test.values) + (1 - best_w) * ridge_full.predict(X_test.values)
print(f"test prediction range at natural scale: [{pred.min():.0f}, {pred.max():.0f}]")

# primary: natural scale (per-fold median peaks at s=1.0)
pd.DataFrame({"Id": test["Id"], "target": pred}).to_csv("submission_v7.csv", index=False)
print(f"saved submission_v7.csv  (range ±{np.abs(pred).max():.0f})")

# optional second entry: amplified to the proven ±800 leaderboard sweet spot
s800 = 800.0 / np.abs(pred).max()
pred_800 = s800 * pred
pd.DataFrame({"Id": test["Id"], "target": pred_800}).to_csv("submission_v7_sweet800.csv", index=False)
print(f"saved submission_v7_sweet800.csv  (scale {s800:.2f}, range ±{np.abs(pred_800).max():.0f})")

test prediction range at natural scale: [-413, 398]
saved submission_v7.csv  (range ±413)
saved submission_v7_sweet800.csv  (scale 1.94, range ±800)


## Results summary

| Model (clip ±2000, raw15 + bounded x9³) | per-fold mean | **median** | min |
|---|---|---|---|
| Ridge α=1000 | 0.0308 | 0.0355 | +0.012 |
| **RandomForest 400** | **0.0336** | **0.0440** | **+0.015** |
| ExtraTrees 400 | 0.0313 | 0.0402 | +0.012 |
| HistGBM d3 | 0.0132 | 0.0178 | −0.031 |
| HistGBM d2 | 0.0171 | 0.0154 | −0.012 |

**Selected: pure RandomForest 400** (blending Ridge in only lowered the median; all folds positive
either way).

- The per-fold median of **0.0440** beats every prior offline result (v3 blend 0.0399, v6 cube 0.0411)
  with a single model, one engineered feature, and no two-stage machinery.
- The scale sweep peaks exactly at s=1.0 — the model's natural ±465 output range is already
  CV-optimal; no shrinking needed.
- `submission_v7.csv` (±413 test range) is the CV-honest primary;
  `submission_v7_sweet800.csv` (±800) tests whether the leaderboard's historical preference for the
  ±800 range transfers to this backbone.